# Task 1 — tabular supervised classification

**Вход:** `dataset/task1/train.csv`, `dataset/task1/test_features.csv`

**Выход:** `outputs/task1_predictions.csv` 



## 1. Настройка

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings("ignore")

# --- Пути (относительно корня проекта) ---
ROOT = Path.cwd().parents[0] if Path.cwd().name == "your-sollution" else Path.cwd()
TRAIN_PATH = ROOT / "dataset" / "task1" / "train.csv"
TEST_PATH = ROOT / "dataset" / "task1" / "test_features.csv"
OUT_PATH = ROOT / "outputs" / "task1_predictions.csv"

# --- Константы ---
TARGET = "prediction"
ID = "patient_id"

# Признаки-утечки: исключаются везде.
DROP_COLS = [ID, "Статус_глюкозы", "Доклинический_риск"]

RANDOM_STATE = 42

print("ROOT:", ROOT)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("OUT_PATH:", OUT_PATH)

## 2. Загрузка данных (шаги 01-02)

In [ ]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Не найден train: {TRAIN_PATH}")
if not TEST_PATH.exists():
    raise FileNotFoundError(f"Не найден test: {TEST_PATH}")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("train:", train.shape, "| test:", test.shape)
print("Баланс классов:", train[TARGET].value_counts(normalize=True).round(3).to_dict())

## 3. Матрицы признаков (единая точка правды)

In [ ]:
missing = [c for c in DROP_COLS + [TARGET] if c not in train.columns]
if missing:
    raise KeyError(f"В train отсутствуют колонки: {missing}")

X = train.drop(columns=DROP_COLS + [TARGET])
y = train[TARGET].astype(int)

drop_in_test = [c for c in DROP_COLS if c in test.columns]
X_test = test.drop(columns=drop_in_test)[X.columns]
test_ids = test[ID].values

assert list(X.columns) == list(X_test.columns), "колонки X и X_test расходятся"
print("X:", X.shape, "| X_test:", X_test.shape)

## 4. Удаление выбросов (шаг 03)

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns
continuous = [c for c in numeric_cols if X[c].nunique(dropna=True) > 2]

q1 = X[continuous].quantile(0.25)
q3 = X[continuous].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

in_bounds = ((X[continuous] >= lower) & (X[continuous] <= upper)).all(axis=1)
print(f"Удалено выбросов (IQR, k=1.5): {int((~in_bounds).sum())} из {len(X)}")

X = X.loc[in_bounds].reset_index(drop=True)
y = y.loc[in_bounds].reset_index(drop=True)
print("После очистки:", X.shape)

## 5. Сплит train / val / test (шаг 06)

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_val, X_holdout, y_val, y_holdout = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE
)
print(
    f"train={X_train.shape[0]} | val={X_val.shape[0]} | test={X_holdout.shape[0]}"
)

## 6. Пропуски средним + StandardScaler (шаги 04-05)

Импутер и scaler обучаются только на train — без утечки.

In [ ]:
imputer = SimpleImputer(strategy="mean").fit(X_train)

def apply_imputer(imp, df):
    return pd.DataFrame(imp.transform(df), columns=df.columns, index=df.index)

X_train = apply_imputer(imputer, X_train)
X_val = apply_imputer(imputer, X_val)
X_holdout = apply_imputer(imputer, X_holdout)
X_test = apply_imputer(imputer, X_test)
print("Пропуски заполнены средним (fit на train)")

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_holdout = scaler.transform(X_holdout)
X_test = scaler.transform(X_test)
print("Данные нормализованы через StandardScaler")

## 7. Модель: 1 слой, 32 нейрона, ReLU, 150 эпох (шаг 07)

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    max_iter=150,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
print(
    f"MLP: hidden_layer_sizes=(32,), activation='relu', "
    f"max_iter=150 (факт. эпох: {model.n_iter_})"
)

proba_val = model.predict_proba(X_val)[:, 1]
proba_hold = model.predict_proba(X_holdout)[:, 1]
print(f"val  ROC-AUC: {roc_auc_score(y_val, proba_val):.4f}")
print(f"test ROC-AUC: {roc_auc_score(y_holdout, proba_hold):.4f}")

## 8. Подбор порога на val (дисбаланс классов)

`MLPClassifier` не поддерживает `class_weight`, поэтому вместо порога 0.5
подбираем порог по val, максимизируя F1.

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.01)
scores = [f1_score(y_val, (proba_val >= t).astype(int)) for t in thresholds]
best_thr = float(thresholds[int(np.argmax(scores))])
print(
    f"Порог по val: {best_thr:.2f} (val F1={max(scores):.4f}, "
    f"default 0.5 F1={f1_score(y_val, (proba_val >= 0.5).astype(int)):.4f})"
)

pred_hold = (proba_hold >= best_thr).astype(int)
print(f"test F1@thr={best_thr:.2f}: {f1_score(y_holdout, pred_hold):.4f}")

## 9. Предсказание и сохранение (шаги 08-09)

In [ ]:
proba_test = model.predict_proba(X_test)[:, 1]
preds = (proba_test >= best_thr).astype(int)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out = pd.DataFrame({ID: test_ids, TARGET: preds})
out.to_csv(OUT_PATH, index=False)

print(f"Сохранено: {OUT_PATH}")
print(out[TARGET].value_counts().to_string())

## 10. Проверка выходного контракта

In [ ]:
expected = pd.read_csv(TEST_PATH)
out_check = pd.read_csv(OUT_PATH)

assert len(out_check) == len(expected), "число строк не совпадает с test"
assert list(out_check.columns) == [ID, TARGET], "неверные колонки"
assert (out_check[ID].values == expected[ID].values).all(), "порядок ID нарушен"
assert out_check[TARGET].notna().all(), "есть пропуски в prediction"
assert set(out_check[TARGET].unique()) <= {0, 1}, "prediction вне {0, 1}"

print("[OK] Контракт соблюдён: строк =", len(out_check))
print(out_check.head())